# Notebook Huấn Luyện & Đánh Giá Mô Hình RQ2 (Viral Regression JP)

**Đề tài:** Phân tích mức độ ảnh hưởng của cảm xúc bình luận tới độ lan tỏa video YouTube tại Nhật Bản.  
**Tác giả:** Hà (Nhóm 6 - ADY201m)  

Nội dung notebook:
1. Nạp và kiểm tra tập dữ liệu `rq2_video_sentiment_JP.csv` (1,018 video hoàn chỉnh)
2. Mô hình Hồi quy Tuyến tính OLS (Kiểm định tham số: p-value, 95% CI)
3. Mô hình Ridge Regression (Chuẩn hóa L2)
4. Mô hình Random Forest Regressor & Đánh giá 5-Fold Cross Validation
5. Trực quan hóa ma trận nhầm lẫn cảm xúc RoBERTa (150 mẫu kiểm chứng)


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Load dataset
df = pd.read_csv('../data/processed/rq2_video_sentiment_JP.csv')
valid_df = df[df['view_count'] > 0].copy()
print(f'Tổng số video hợp lệ: {len(valid_df)}')
valid_df[['view_count', 'negative_ratio', 'positive_ratio', 'log_view_count']].describe()


## 1. Mô Hình Hồi Quy Tuyến Tính OLS (statsmodels)
Kiểm định giả thuyết tham số H0 (beta_negative = 0) so với H1 (beta_negative != 0).


In [ ]:
target_col = 'log_view_count'
y = valid_df[target_col].values

num_features = [
    'negative_ratio', 'positive_ratio', 'like_to_view_ratio',
    'comment_to_view_ratio', 'upload_hour_jst', 'is_weekend'
]
cat_dummies = pd.get_dummies(valid_df['category_id'].astype(str), prefix='cat', drop_first=True)
X = pd.concat([valid_df[num_features].fillna(0), cat_dummies], axis=1).astype(float)

X_const = sm.add_constant(X)
ols_model = sm.OLS(y, X_const).fit()
print(ols_model.summary())


## 2. Mô Hình Ridge Regression & Random Forest (5-Fold CV)
So sánh hiệu năng giữa mô hình tuyến tính chuẩn hóa L2 và mô hình phi tuyến quần thể cây.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
print(f'Ridge Test R2: {r2_score(y_test, y_pred_ridge):.4f}, MAE: {mean_absolute_error(y_test, y_pred_ridge):.4f}')

rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f'Random Forest Test R2: {r2_score(y_test, y_pred_rf):.4f}, MAE: {mean_absolute_error(y_test, y_pred_rf):.4f}')

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rf = cross_val_score(rf, X, y, cv=kf, scoring='r2')
print(f'Random Forest 5-Fold CV R2: {cv_rf.mean():.4f} +/- {cv_rf.std():.4f}')


## 3. Ma Trận Nhầm Lẫn & Kiểm Chứng Cảm Xúc (XLM-RoBERTa)
Đánh giá trên tập Ground Truth 150 mẫu tiếng Nhật (`reports/assets/rq2/jp_sentiment_validation.csv`).


In [ ]:
val_df = pd.read_csv('../reports/assets/rq2/jp_sentiment_validation.csv')
ct = pd.crosstab(val_df['true_label'], val_df['predicted_label'], margins=True)
print('Confusion Matrix:')
print(ct)
